# Task 1 — PilotNet (NVIDIA 2016) on CARLA Steering Dataset
**Name:**MD Ayaan| **Roll No:** 25155053 

---

ok this task was probably the most interesting one. the idea is behavioral cloning — you show a network a bunch of (image, steering_angle) pairs and it learns to steer on its own.

i read the original NVIDIA paper for this:
> Bojarski et al., "End to End Learning for Self-Driving Cars", NVIDIA 2016

read it like 3 times before i understood what they were actually doing lol. the key insight is there's no manual feature engineering — the network learns directly from pixels to steering angle.

this is REGRESSION not classification. took me a while to internalize that difference.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using device:', device)

## getting the dataset from kaggle

In [ ]:
# kaggle import — copy pasted from the dataset page
# dataset: https://www.kaggle.com/datasets/...

# from kaggle.api.kaggle_api_extended import KaggleApi
# api = KaggleApi()
# api.authenticate()
# api.dataset_download_files('USERNAME/carla-steering-dataset', path='./data', unzip=True)

# already downloaded, commenting out
DATA_PATH = './data/carla_steering'

## Loading and Exploring the Dataset

first thing: print the columns because i got a KeyError the first time from wrong column name 💀

In [ ]:
# spent like 20 mins debugging a KeyError because i assumed wrong column names
# always print columns first
df = pd.read_csv(os.path.join(DATA_PATH, 'data.csv'))
print(df.head())
print()

print('columns:', df.columns.tolist())
print('shape:', df.shape)

In [ ]:
# check steering angle distribution — i expected lots of near-zero values
plt.figure(figsize=(10, 4))
plt.hist(df['steering'], bins=100, color='steelblue', edgecolor='black')
plt.title('steering angle distribution (raw)')
plt.xlabel('steering angle')
plt.ylabel('count')
plt.savefig('steering_dist.png', dpi=100, bbox_inches='tight')
plt.show()





print('mean:', df['steering'].mean())
print('std:', df['steering'].std())
print('min/max:', df['steering'].min(), '/', df['steering'].max())

## Fixing the Data Imbalance

so like 80% of the data is near-zero steering (straight driving). if i train on this as-is the model will just learn to always output ~0 and get decent loss without actually learning to turn.

fix: undersample the straight-driving frames. keep all turning samples, keep only 30% of near-zero samples.

also: when i do horizontal flip augmentation later i need to NEGATE the steering angle. took me an embarrassingly long time to realize this. if the car was turning left in the original and i flip the image, it should now be turning right → so steering = -steering.

In [ ]:
threshold = 0.1
keep_fraction = 0.3



straight = df[df['steering'].abs() < threshold]
turning  = df[df['steering'].abs() >= threshold]



straight_sampled = straight.sample(frac=keep_fraction, random_state=42)
df_balanced = pd.concat([straight_sampled, turning]).reset_index(drop=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print('original size:', len(df))
print('after balancing:', len(df_balanced))



plt.figure(figsize=(10, 4))
plt.hist(df_balanced['steering'], bins=100, color='coral', edgecolor='black')
plt.title('steering angle distribution (after balancing)')
plt.xlabel('steering angle')
plt.ylabel('count')
plt.savefig('steering_dist_balanced.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# 80/20 split manually — already shuffled above
split_idx = int(0.8 * len(df_balanced))
train_df = df_balanced[:split_idx].reset_index(drop=True)
val_df   = df_balanced[split_idx:].reset_index(drop=True)



print('train:', len(train_df))
print('val:', len(val_df))

## Dataset Class

preprocessing follows the paper exactly:
- crop top 35% (removes sky / trees that aren't relevant to steering)
- resize to 66×200 (the paper's input size)
- convert BGR→YUV (YUV separates luminance from color, paper uses this)

the Y channel in YUV is brightness — so brightness augmentation only on Y is cleaner than doing it on all RGB channels.

In [ ]:
class SteeringDataset(Dataset):
    
    
    
    def __init__(self, df, img_root, augment=False):
        self.df = df
        self.img_root = img_root
        self.augment = augment
        self.target_h = 66
        self.target_w = 200

    
    
    
    def __len__(self):
        return len(self.df)

    
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row['image_path'])
        img = cv2.imread(img_path)

        
        if img is None:
            # sometimes image fails to load, just return zeros
            img = np.zeros((self.target_h, self.target_w, 3), dtype=np.uint8)

        
        
        # crop out sky — top 35% is usually not useful
        crop_top = int(img.shape[0] * 0.35)
        img = img[crop_top:, :]

        img = cv2.resize(img, (self.target_w, self.target_h))

        
        
        # YUV color space from the paper
        img = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)

        steering = float(row['steering'])

        if self.augment:
            # horizontal flip — MUST negate steering angle
            # if turning left + flip image = turning right
            if np.random.random() > 0.5:
                img = cv2.flip(img, 1)
                steering = -steering

           
           
           
           
           # brightness aug on Y channel only (luminance)
            if np.random.random() > 0.5:
                delta = np.random.uniform(-30, 30)
                img[:, :, 0] = np.clip(img[:, :, 0].astype(np.int32) + delta, 0, 255).astype(np.uint8)

        img = img.astype(np.float32) / 255.0
        img = torch.FloatTensor(img).permute(2, 0, 1)  # HWC ti CHW

        return img, torch.FloatTensor([steering])

In [ ]:
IMG_ROOT = DATA_PATH

train_dataset = SteeringDataset(train_df, IMG_ROOT, augment=True)
val_dataset   = SteeringDataset(val_df,   IMG_ROOT, augment=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2)

# sanity check
imgs, labels = next(iter(train_loader))
print('image batch:', imgs.shape)   # should be [64, 3, 66, 200]
print('label batch:', labels.shape) # should be [64, 1]

## PilotNet Architecture

from the NVIDIA 2016 paper (Bojarski et al.):
- 5 conv layers with specific filter sizes and strides from the paper
- 4 FC layers
- output: single value (steering angle)

this is **regression** — continuous output, not discrete classes. that means:
- MSE loss, not CrossEntropy
- no softmax at the end
- output neuron has no activation (can be negative for left turns)

the first 3 conv layers use stride=2 to downsample spatially instead of pooling. the last 2 use 3x3 with stride=1.

In [ ]:
class PilotNet(nn.Module):
    def __init__(self):
        super(PilotNet, self).__init__()

        
        
        
        # exact architecture from the nvidia paper
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=5, stride=2),
            nn.ELU(),

            
            
            nn.Conv2d(24, 36, kernel_size=5, stride=2),
            nn.ELU(),

            
            
            nn.Conv2d(36, 48, kernel_size=5, stride=2),
            nn.ELU(),

            nn.Conv2d(48, 64, kernel_size=3),
            nn.ELU(),

            
            nn.Conv2d(64, 64, kernel_size=3),
            nn.ELU(),
        )

        
        
        # flat size after conv layers on 66x200 input = 1152
        # i verified this with the dummy forward pass below
        
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1152, 100),
            nn.ELU(),
            nn.Dropout(0.2),

            nn.Linear(100, 50),
            nn.ELU(),

            nn.Linear(50, 10),
            nn.ELU(),

            nn.Linear(10, 1)
            # no activation at output — steering can be negative
        )

    
    
    
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [ ]:
# check flat size before hardcoding 1152
# i always do this after defining a new arch — saves debugging time later

model = PilotNet().to(device)



dummy = torch.zeros(1, 3, 66, 200).to(device)
with torch.no_grad():
    conv_out = model.conv_layers(dummy)


print('conv output shape:', conv_out.shape)
print('flat size:', conv_out.numel())  # should be 1152

with torch.no_grad():
    out = model(dummy)
print('final output:', out.shape)  # should be [1, 1]



total = sum(p.numel() for p in model.parameters())
print(f'total params: {total:,}')

## Training

using MSELoss because this is regression. MSE = mean of (pred - true)² which penalizes large errors more.

ReduceLROnPlateau scheduler — reduces lr when val loss stops improving. this is better than StepLR here because regression loss doesn't always follow a predictable decay pattern.

In [ ]:
model = PilotNet().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# reduce lr if val loss stops improving for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

train_losses = []
val_losses   = []

NUM_EPOCHS = 30
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for batch_imgs, batch_labels in train_loader:
        batch_imgs   = batch_imgs.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        preds = model(batch_imgs)
        loss  = criterion(preds, batch_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_imgs, batch_labels in val_loader:
            batch_imgs   = batch_imgs.to(device)
            batch_labels = batch_labels.to(device)
            preds = model(batch_imgs)
            val_loss += criterion(preds, batch_labels).item()

    val_loss = val_loss / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'pilotnet_best.pth')

    if (epoch+1) % 5 == 0:
        print(f'epoch {epoch+1}/{NUM_EPOCHS} | train: {train_loss:.4f} | val: {val_loss:.4f} | rmse: {val_loss**0.5:.4f}')

## Results

plotting stuff — used AI help for the scatter plot formatting, i wasn't sure how to do the diagonal reference line and the alpha transparency.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='train', color='steelblue')
plt.plot(val_losses,   label='val',   color='orange')
plt.xlabel('epoch')
plt.ylabel('MSE loss')
plt.title('pilotnet training curves')
plt.legend()
plt.grid(True)
plt.savefig('pilotnet_loss_curves.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# load best model
model.load_state_dict(torch.load('pilotnet_best.pth', map_location=device))
model.eval()

# collect all predictions on val set
all_preds = []
all_true  = []

with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(device)).cpu().numpy().flatten()
        all_preds.extend(preds)
        all_true.extend(labels.numpy().flatten())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

# scatter — used AI for the plot syntax here
plt.figure(figsize=(7, 7))
plt.scatter(all_true, all_preds, alpha=0.2, s=5, color='steelblue')
plt.plot([-1, 1], [-1, 1], color='red', linestyle='--', label='perfect prediction')
plt.xlabel('true steering')
plt.ylabel('predicted steering')
plt.title('predicted vs true steering angle')
plt.legend()
plt.grid(True)
plt.savefig('pilotnet_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

mse = np.mean((all_preds - all_true)**2)
mae = np.mean(np.abs(all_preds - all_true))
print(f'MSE:  {mse:.4f}')
print(f'RMSE: {mse**0.5:.4f}')
print(f'MAE:  {mae:.4f}')

In [ ]:
# sample image grid with angle overlay
sample_imgs, sample_labels = next(iter(val_loader))
with torch.no_grad():
    sample_preds = model(sample_imgs.to(device)).cpu().numpy().flatten()
sample_labels = sample_labels.numpy().flatten()

n = 10
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(n):
    img_np = sample_imgs[i].permute(1, 2, 0).numpy()
    img_np = (img_np * 255).astype(np.uint8)
    img_rgb = cv2.cvtColor(img_np, cv2.COLOR_YUV2RGB)  # convert back for display

    axes[i].imshow(img_rgb)
    axes[i].set_title(f'true: {sample_labels[i]:.3f}\npred: {sample_preds[i]:.3f}', fontsize=8)
    axes[i].axis('off')

plt.suptitle('sample predictions')
plt.tight_layout()
plt.savefig('pilotnet_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

## Summary

key things i learned:

1. **regression vs classification** — continuous output, MSE loss, no softmax. this was confusing at first because all the examples i'd seen used classification.

2. **data imbalance in regression** — even for regression data can be imbalanced. straight-driving frames were majority so i undersampled them.

3. **flip → negate label** — this is unique to regression tasks with spatial meaning. horizontal flip reverses the direction so steering sign must flip too.

4. **YUV colorspace** — separating luminance lets you do cleaner brightness augmentation on just the Y channel.

overall the model learns the general steering pattern but isn't perfect. real self-driving systems obviously use way more data and sensors, but the core idea here is solid.